# ModernBERT-base Fine-Tuning — Context Depth Classifier

Fine-tunes `answerdotai/ModernBERT-base` as a **4-class skill depth classifier** for the ConFit employee onboarding pipeline.

| Class | Depth Label | Score | Meaning |
|---|---|---|---|
| 0 | surface | 0.2 | Mentioned/listed only, familiar with, coursework |
| 1 | regular | 0.5 | Regular use in job tasks, "used X to do Y" |
| 2 | deep | 0.8 | Optimized, led, designed, production use |
| 3 | expert | 1.0 | Architected, built from scratch at scale |

**Two dataset sources (used together):**
1. **Existing** — `resume_evidence.jsonl` from NB2 (upload as Kaggle dataset — see Section 1)
2. **Optional** — GPT-4o-mini batch extraction on raw Kaggle resumes for expansion

**Training strategy:** Freeze transformer blocks 0–20, train blocks 21–22 + classifier head (~14M / 149M params)

| Setting | Value |
|---|---|
| Base model | `answerdotai/ModernBERT-base` |
| Task | Sequence Classification (4 classes) |
| Frozen | Embedding + blocks 0–20 |
| Trainable | Blocks 21–22, LayerNorm, classifier head |
| Epochs | 5 |
| Batch size | 32 |
| Learning rate | 2e-4 |
| Max seq len | 128 |
| Runtime | **Kaggle P100/T4 (~35-40 min)** |

## Before Running on Kaggle
1. Go to **Add Data** → **Your Datasets** → upload your `resume_evidence.jsonl` (from NB2 outputs) as a new dataset named `confit-nb2-evidence`.
2. The Snehaanbhawal resume dataset (`snehaanbhawal/resume-dataset`) should already be added via **Add Data** → **Datasets** if you plan to run GPT expansion.
3. Set **Accelerator → GPU P100 or T4** in Notebook Settings.
4. Enable **Internet** in Settings (needed for downloading `answerdotai/ModernBERT-base` from HuggingFace).

## Section 0 — Install Dependencies

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", *args], check=True)

# On Kaggle, torch / numpy / pandas / sklearn / matplotlib / seaborn are pre-installed.
# We only upgrade the HuggingFace stack + openai + nest_asyncio.
pip("install", "-q",
    "transformers>=4.40.0",
    "datasets",
    "accelerate",
    "openai",
    "nest_asyncio",
)

print("✓ Dependencies ready.")

In [ ]:
import os
import json
import asyncio
import warnings
import random
from pathlib import Path
from collections import Counter

import nest_asyncio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
)
from openai import AsyncOpenAI

nest_asyncio.apply()  # allow asyncio.run() inside Jupyter
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Environment detection ──────────────────────────────────────────────────────
IS_KAGGLE = os.path.exists("/kaggle/input")

# ── Input dataset slugs (edit these to match your Kaggle dataset names) ────────
# Upload resume_evidence.jsonl from NB2 as a Kaggle dataset first:
#   Kaggle → Datasets → New Dataset → name it "confit-nb2-evidence"
KAGGLE_NB2_SLUG    = "confit-nb2-evidence"          # your uploaded dataset slug
KAGGLE_RESUME_SLUG = "snehaanbhawal/resume-dataset"  # public Kaggle resume dataset

# ── Paths ──────────────────────────────────────────────────────────────────────
if IS_KAGGLE:
    WORKING_DIR  = Path("/kaggle/working")
    # resume_evidence.jsonl — uploaded by you as a Kaggle dataset
    RESUME_EVIDENCE_PATH = Path(f"/kaggle/input/{KAGGLE_NB2_SLUG}/resume_evidence.jsonl")
    # Raw Kaggle resume CSV — available if you added snehaanbhawal/resume-dataset
    KAGGLE_RESUME_CSV    = Path(f"/kaggle/input/resume-dataset/Resume/Resume.csv")
    DATASET_OUT          = WORKING_DIR / "depth_classifier"
    MODEL_SAVE_PATH      = WORKING_DIR / "depth-modernbert"
else:
    # Local fallback (notebook lives in machine_learning/)
    WORKING_DIR          = Path(".")
    RESUME_EVIDENCE_PATH = WORKING_DIR / "data/nb2_outputs/resume_evidence.jsonl"
    KAGGLE_RESUME_CSV    = None
    DATASET_OUT          = WORKING_DIR / "data/depth_classifier"
    MODEL_SAVE_PATH      = WORKING_DIR / "models/depth-modernbert"

DATASET_OUT.mkdir(parents=True, exist_ok=True)
MODEL_SAVE_PATH.mkdir(parents=True, exist_ok=True)

DATASET_CSV = DATASET_OUT / "depth_classifier_dataset.csv"

# ── Model config ───────────────────────────────────────────────────────────────
BASE_MODEL_NAME = "answerdotai/ModernBERT-base"
NUM_LABELS      = 4
MAX_SEQ_LEN     = 128

# ── Training hyperparams ────────────────────────────────────────────────────────
LEARNING_RATE   = 2e-4
EPOCHS          = 5
BATCH_SIZE      = 32
WARMUP_RATIO    = 0.10
WEIGHT_DECAY    = 0.01
MAX_GRAD_NORM   = 1.0

# ── Label maps ─────────────────────────────────────────────────────────────────
DEPTH_STR_TO_INT   = {"surface": 0, "regular": 1, "deep": 2, "expert": 3}
DEPTH_FLOAT_TO_INT = {0.2: 0, 0.5: 1, 0.8: 2, 1.0: 3}
INT_TO_DEPTH_FLOAT = {0: 0.2, 1: 0.5, 2: 0.8, 3: 1.0}
CLASS_NAMES        = ["surface (0.2)", "regular (0.5)", "deep (0.8)", "expert (1.0)"]

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(f"Environment  : {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Working dir  : {WORKING_DIR}")
print(f"NB2 evidence : {RESUME_EVIDENCE_PATH}  (exists={RESUME_EVIDENCE_PATH.exists()})")
print(f"Dataset out  : {DATASET_OUT}")
print(f"Model path   : {MODEL_SAVE_PATH}")

## Section 1 — Load Existing Labeled Data (`resume_evidence.jsonl`)

NB2 already ran NVIDIA LLM extraction over all resumes and stored results with `depth_signal` labels (`surface / regular / deep / expert`). We load this as the **primary dataset source** — no extra API cost needed.

Then in Section 2 we optionally expand with GPT-4o-mini on the raw Kaggle resumes for more `surface` and `expert` examples.

In [ ]:
existing_records = []

with open(RESUME_EVIDENCE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            continue

        for skill_item in record.get("skills", []):
            evidence    = skill_item.get("evidence", "").strip()
            depth_str   = skill_item.get("depth_signal", "").lower().strip()
            skill_name  = skill_item.get("skill", "").strip()

            # Skip if evidence is empty or just a copy of the skill name
            if not evidence or evidence.lower() == skill_name.lower():
                continue
            # Skip unknown depth labels
            if depth_str not in DEPTH_STR_TO_INT:
                continue
            # Skip very short sentences (< 8 words)
            if len(evidence.split()) < 8:
                continue

            existing_records.append({
                "text":   evidence,
                "label":  DEPTH_STR_TO_INT[depth_str],
                "source": "nb2_evidence",
            })

df_existing = pd.DataFrame(existing_records).drop_duplicates(subset=["text"])

print(f"Loaded {len(df_existing):,} records from resume_evidence.jsonl")
print("\nLabel distribution (existing NB2 data):")
dist = df_existing["label"].value_counts().sort_index()
for lbl, cnt in dist.items():
    pct = cnt / len(df_existing) * 100
    bar = "█" * int(pct / 2)
    print(f"  Class {lbl} ({CLASS_NAMES[lbl]:>14s}): {cnt:>4d}  ({pct:4.1f}%)  {bar}")

## Section 2 — GPT-4o-mini Batch Extraction (Optional Expansion)

Run this section if the existing NB2 data is insufficient (especially for `surface` and `expert` classes).

**Cost estimate:** 2400 resumes ÷ 10 per batch = 240 API calls × ~$0.006/call ≈ **$1.50–$2.00**

**Time estimate:** ~30–40 minutes with async calls (concurrency=10)

> **Skip this section** if `df_existing` already has 1500+ well-distributed records. Jump straight to Section 3.

In [ ]:
# ── Config — set RUN_GPT_EXTRACTION = True to run this section ────────────────
RUN_GPT_EXTRACTION = False          # ← flip to True if you need more data
OPENAI_API_KEY     = os.getenv("OPENAI_API_KEY", "sk-...")
# On Kaggle: add your OpenAI key via Notebook Secrets (Settings → Add-ons → Secrets)
# then use: OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
GPT_MODEL          = "gpt-4o-mini"
CONCURRENCY        = 10             # simultaneous requests
MAX_SKILLS_PER_RES = 8

GPT_PROMPT_TEMPLATE = """\
You are a skill evidence extractor.

From the resume below, extract skill usage sentences.
For each skill found, extract the sentence or phrase that best describes
HOW the person used the skill — their actual work context.

Rules:
- Skip if context is just the skill name with no usage description
- Skip soft skills entirely
- Maximum {max_skills} skills per resume
- Depth scale:
    0.2 = mentioned/listed only, exposure, "familiar with", coursework
    0.5 = regular use in job tasks, "used X to do Y"
    0.8 = optimized, led, designed, improved performance, production use
    1.0 = architected, core contributor, expert-level, built from scratch at scale

Return a JSON array ONLY — no markdown, no explanation:
[
  {{
    "skill": "raw skill name",
    "context_sentence": "exact phrase from resume describing usage",
    "depth_class": 0.2 | 0.5 | 0.8 | 1.0
  }}
]

Resume:
{resume_text}"""


async def extract_from_resume(client: AsyncOpenAI, resume_text: str, sem: asyncio.Semaphore):
    """Call GPT on a single resume and return parsed list of skill dicts."""
    prompt = GPT_PROMPT_TEMPLATE.format(
        max_skills=MAX_SKILLS_PER_RES,
        resume_text=resume_text[:6000],
    )
    async with sem:
        try:
            response = await client.chat.completions.create(
                model=GPT_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=1200,
            )
            raw_json = response.choices[0].message.content.strip()
            if raw_json.startswith("```"):
                raw_json = "\n".join(raw_json.split("\n")[1:-1])
            return json.loads(raw_json)
        except Exception as e:
            print(f"  GPT error: {e}")
            return []


async def run_gpt_extraction(resumes: list[str]) -> list[dict]:
    client = AsyncOpenAI(api_key=OPENAI_API_KEY)
    sem = asyncio.Semaphore(CONCURRENCY)
    tasks = [extract_from_resume(client, r, sem) for r in resumes]
    results = []
    total = len(tasks)
    for i, coro in enumerate(asyncio.as_completed(tasks)):
        batch_result = await coro
        results.extend(batch_result)
        if (i + 1) % 50 == 0:
            print(f"  Progress: {i+1}/{total} resumes processed, {len(results)} records so far")
    return results


if RUN_GPT_EXTRACTION:
    # On Kaggle, the resume CSV is already mounted — no kagglehub download needed
    if IS_KAGGLE and KAGGLE_RESUME_CSV and KAGGLE_RESUME_CSV.exists():
        print(f"Loading Kaggle resume dataset from: {KAGGLE_RESUME_CSV}")
        raw_df = pd.read_csv(KAGGLE_RESUME_CSV)
    else:
        # Local fallback — download via kagglehub (needs KAGGLE_USERNAME/KEY env vars)
        import kagglehub
        kaggle_path = kagglehub.dataset_download("snehaanbhawal/resume-dataset")
        raw_df = pd.read_csv(Path(kaggle_path) / "Resume" / "Resume.csv")

    resumes_raw = raw_df["Resume_str"].dropna().tolist()
    print(f"  Loaded {len(resumes_raw)} resumes")

    print(f"Running GPT extraction (concurrency={CONCURRENCY})...")
    gpt_raw = asyncio.run(run_gpt_extraction(resumes_raw))
    print(f"  Done — {len(gpt_raw)} total skill records")

    gpt_raw_path = DATASET_OUT / "gpt_raw_records.json"
    with open(gpt_raw_path, "w") as f:
        json.dump(gpt_raw, f)
    print(f"  Raw GPT records saved to {gpt_raw_path}")
else:
    gpt_raw = []
    print("GPT extraction skipped (RUN_GPT_EXTRACTION=False).")

## Section 3 — Post-Process, Merge & Build Final Dataset

Merge existing NB2 evidence with any GPT-extracted records. Apply quality filters:
- Remove sentences < 8 words
- Deduplicate on exact text
- Remove records where text == skill name only

In [ ]:
# ── Process GPT records (if any) ───────────────────────────────────────────────
gpt_records = []
for item in gpt_raw:
    text  = item.get("context_sentence", "").strip()
    skill = item.get("skill", "").strip()
    dc    = item.get("depth_class")

    if not text or text.lower() == skill.lower():
        continue
    if dc not in DEPTH_FLOAT_TO_INT:
        continue
    if len(text.split()) < 8:
        continue

    gpt_records.append({
        "text":   text,
        "label":  DEPTH_FLOAT_TO_INT[dc],
        "source": "gpt_extraction",
    })

df_gpt = pd.DataFrame(gpt_records) if gpt_records else pd.DataFrame(columns=["text", "label", "source"])

# ── Merge sources ──────────────────────────────────────────────────────────────
df_all = pd.concat([df_existing, df_gpt], ignore_index=True)
df_all = df_all.drop_duplicates(subset=["text"]).reset_index(drop=True)

print(f"Total records after merge & dedup: {len(df_all):,}")
print(f"  From NB2 evidence : {(df_all['source'] == 'nb2_evidence').sum():,}")
print(f"  From GPT          : {(df_all['source'] == 'gpt_extraction').sum():,}")

# ── Final label distribution ───────────────────────────────────────────────────
print("\nFinal class distribution:")
dist = df_all["label"].value_counts().sort_index()
for lbl, cnt in dist.items():
    pct = cnt / len(df_all) * 100
    bar = "█" * int(pct / 2)
    print(f"  Class {lbl} ({CLASS_NAMES[lbl]:>14s}): {cnt:>4d}  ({pct:4.1f}%)  {bar}")

# ── Visualize ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Bar chart
colors = ["#78c5e0", "#5ba85f", "#e07b54", "#9b59b6"]
axes[0].bar(CLASS_NAMES, dist.values, color=colors, edgecolor="black", linewidth=0.8)
axes[0].set_title("Class Distribution", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Count")
axes[0].set_xlabel("Depth Class")
for i, v in enumerate(dist.values):
    axes[0].text(i, v + 10, str(v), ha="center", fontweight="bold")

# Pie chart
axes[1].pie(dist.values, labels=CLASS_NAMES, colors=colors, autopct="%1.1f%%",
            startangle=90, wedgeprops=dict(edgecolor="black", linewidth=0.8))
axes[1].set_title("Class Share", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.savefig(DATASET_OUT / "class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Save full dataset ──────────────────────────────────────────────────────────
df_all[["text", "label"]].to_csv(DATASET_CSV, index=False)
print(f"\n✓ Saved to {DATASET_CSV}")

## Section 4 — Train / Val / Test Split

Stratified 80 / 10 / 10 split preserving class ratios in each partition.

In [ ]:
df_clean = pd.read_csv(DATASET_CSV)

train_df, temp_df = train_test_split(
    df_clean, test_size=0.20, stratify=df_clean["label"], random_state=RANDOM_SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=RANDOM_SEED
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Split sizes:")
print(f"  Train : {len(train_df):,} ({len(train_df)/len(df_clean)*100:.0f}%)")
print(f"  Val   : {len(val_df):,}   ({len(val_df)/len(df_clean)*100:.0f}%)")
print(f"  Test  : {len(test_df):,}   ({len(test_df)/len(df_clean)*100:.0f}%)")

print("\nTrain class distribution:")
for lbl, cnt in train_df["label"].value_counts().sort_index().items():
    print(f"  Class {lbl}: {cnt}")

# Save splits
train_df.to_csv(DATASET_OUT / "train.csv", index=False)
val_df.to_csv(DATASET_OUT / "val.csv", index=False)
test_df.to_csv(DATASET_OUT / "test.csv", index=False)
print("\n✓ Splits saved.")

## Section 5 — Tokenization & HuggingFace Dataset Preparation

Load splits into HuggingFace `Dataset` objects and tokenize with the ModernBERT tokenizer.
`max_length=128` — context sentences rarely exceed 64 tokens, 128 gives headroom.

In [ ]:
print(f"Loading tokenizer: {BASE_MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

def tokenize_batch(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_SEQ_LEN,
    )

def df_to_hf_dataset(df: pd.DataFrame) -> Dataset:
    ds = Dataset.from_dict({"text": df["text"].tolist(), "labels": df["label"].tolist()})
    ds = ds.map(tokenize_batch, batched=True, batch_size=256, desc="Tokenizing")
    ds = ds.remove_columns(["text"])
    ds.set_format("torch")
    return ds

print("Tokenizing splits...")
train_ds = df_to_hf_dataset(train_df)
val_ds   = df_to_hf_dataset(val_df)
test_ds  = df_to_hf_dataset(test_df)

print(f"\n✓ Train dataset : {len(train_ds):,} samples | features: {train_ds.features}")
print(f"  Val dataset   : {len(val_ds):,} samples")
print(f"  Test dataset  : {len(test_ds):,} samples")

# Inspect one example
sample = train_ds[0]
print(f"\nSample input_ids shape : {sample['input_ids'].shape}")
print(f"Sample attention_mask  : {sample['attention_mask'].sum().item()} real tokens / {MAX_SEQ_LEN} total")

## Section 6 — Load ModernBERT-base & Freeze Layers

**What gets trained (Option A — Last 2 Blocks):**

```
Embedding layer         → FROZEN
Transformer blocks 0–20 → FROZEN   (21 of 22 blocks)
Transformer block 21    → TRAINABLE ← last block
Transformer block 22    → TRAINABLE ← last block  
LayerNorm (final)       → TRAINABLE
Classifier head         → TRAINABLE ← new 768→4 head
```

~14M trainable out of 149M total params (~9%).

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_NAME,
    num_labels=NUM_LABELS,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
    ignore_mismatched_sizes=True,   # classifier head will differ from a pretrained task head
)

# ── Identify the total number of transformer layers ────────────────────────────
# ModernBERT-base uses model.model.layers (list of EncoderLayer)
layers = model.model.layers
N_LAYERS = len(layers)
N_FREEZE = N_LAYERS - 2   # freeze all but the last 2
print(f"Total transformer blocks : {N_LAYERS}")
print(f"Freezing blocks 0–{N_FREEZE-1}, training blocks {N_FREEZE}–{N_LAYERS-1}")

# ── Step 1: Freeze everything ──────────────────────────────────────────────────
for param in model.parameters():
    param.requires_grad = False

# ── Step 2: Unfreeze last 2 transformer blocks ─────────────────────────────────
for layer in layers[N_FREEZE:]:
    for param in layer.parameters():
        param.requires_grad = True

# ── Step 3: Unfreeze final LayerNorm ──────────────────────────────────────────
for name, param in model.named_parameters():
    if "final_norm" in name or "post_norm" in name or (
        "norm" in name and "layer" not in name and param.requires_grad is False
        and "model.layers" not in name
    ):
        param.requires_grad = True

# ── Step 4: Unfreeze classifier head (always) ─────────────────────────────────
for param in model.classifier.parameters():
    param.requires_grad = True

# ── Report ─────────────────────────────────────────────────────────────────────
trainable   = sum(p.numel() for p in model.parameters() if p.requires_grad)
total       = sum(p.numel() for p in model.parameters())
frozen      = total - trainable
print(f"\nParameter summary:")
print(f"  Trainable : {trainable:>12,}  ({trainable/total*100:.1f}%)")
print(f"  Frozen    : {frozen:>12,}  ({frozen/total*100:.1f}%)")
print(f"  Total     : {total:>12,}")

model = model.to(device)

## Section 7 — Compute Class Weights

The dataset is imbalanced (regular ~40%, expert ~5%). We compute balanced class weights and pass them to `CrossEntropyLoss` so the model doesn't simply predict the majority class.

In [ ]:
class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2, 3]),
    y=train_df["label"].values,
)
class_weights_tensor = torch.FloatTensor(class_weights_np).to(device)

print("Class weights (balanced):")
for i, (name, w) in enumerate(zip(CLASS_NAMES, class_weights_np)):
    bar = "█" * int(w * 10)
    print(f"  Class {i} ({name:>14s}): {w:.4f}  {bar}")

print(f"\nTensor: {class_weights_tensor}")


# ── Custom Trainer with weighted loss ─────────────────────────────────────────
class WeightedLossTrainer(Trainer):
    """Trainer subclass that uses class-weighted CrossEntropyLoss."""

    def __init__(self, *args, class_weights: torch.Tensor = None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits

        loss_fn = torch.nn.CrossEntropyLoss(weight=self.class_weights)
        loss    = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

## Section 8 — Define Trainer & Training Arguments

Key decisions:
- `evaluation_strategy="epoch"` — evaluate after every epoch to catch overfitting early
- `load_best_model_at_end=True` — auto-restore best checkpoint at finish
- `metric_for_best_model="f1"` — optimize for macro F1, not just accuracy

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro_f1   = f1_score(labels, preds, average="macro", zero_division=0)
    accuracy   = accuracy_score(labels, preds)
    per_class  = f1_score(labels, preds, average=None, zero_division=0)
    metrics = {
        "f1":       round(macro_f1, 4),
        "accuracy": round(accuracy, 4),
    }
    for i, f in enumerate(per_class):
        metrics[f"f1_class_{i}"] = round(float(f), 4)
    return metrics


CHECKPOINT_DIR = DATASET_OUT / "checkpoints"

training_args = TrainingArguments(
    output_dir                  = str(CHECKPOINT_DIR),
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE * 2,
    learning_rate               = LEARNING_RATE,
    warmup_ratio                = WARMUP_RATIO,
    weight_decay                = WEIGHT_DECAY,
    lr_scheduler_type           = "linear",
    max_grad_norm               = MAX_GRAD_NORM,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    greater_is_better           = True,
    fp16                        = torch.cuda.is_available(),  # AMP on GPU
    dataloader_num_workers      = 0,                          # 0 = safe on Windows/Colab
    report_to                   = "none",
    seed                        = RANDOM_SEED,
    logging_steps               = 20,
    logging_first_step          = True,
)

trainer = WeightedLossTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = train_ds,
    eval_dataset     = val_ds,
    tokenizer        = tokenizer,
    compute_metrics  = compute_metrics,
    class_weights    = class_weights_tensor,
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Trainer configured.")
print(f"  Epochs          : {EPOCHS}")
print(f"  Batch size      : {BATCH_SIZE}")
print(f"  Learning rate   : {LEARNING_RATE}")
print(f"  FP16 (AMP)      : {training_args.fp16}")
print(f"  Checkpoint dir  : {CHECKPOINT_DIR}")

## Section 9 — Run Training

Expected runtime: **35–40 minutes on Colab T4** for 5 epochs on ~1400 training samples.

Watch the `eval/f1` column — it should increase each epoch. If it plateaus or drops, early stopping will kick in (patience=2 epochs).

In [ ]:
print("Starting training...")
print("=" * 60)

train_result = trainer.train()

print("\n" + "=" * 60)
print("Training complete!")
print(f"  Total steps        : {train_result.global_step}")
print(f"  Train runtime      : {train_result.metrics['train_runtime']:.0f}s  "
      f"({train_result.metrics['train_runtime']/60:.1f} min)")
print(f"  Samples/second     : {train_result.metrics['train_samples_per_second']:.1f}")
print(f"  Final train loss   : {train_result.metrics.get('train_loss', 'N/A')}")

# ── Plot training curves ───────────────────────────────────────────────────────
log_history = trainer.state.log_history

train_losses = [(e["step"], e["loss"])        for e in log_history if "loss" in e]
eval_metrics = [(e["epoch"], e["eval_f1"], e.get("eval_loss"))
                for e in log_history if "eval_f1" in e]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

if train_losses:
    steps, losses = zip(*train_losses)
    axes[0].plot(steps, losses, color="#5ba85f", linewidth=1.5)
    axes[0].set_title("Training Loss", fontsize=12, fontweight="bold")
    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Loss")
    axes[0].grid(alpha=0.3)

if eval_metrics:
    epochs_e, f1s, eval_losses = zip(*eval_metrics)
    axes[1].plot(epochs_e, f1s, color="#5ba85f", marker="o", linewidth=2, label="Val Macro F1")
    if any(el is not None for el in eval_losses):
        ax2 = axes[1].twinx()
        ax2.plot(epochs_e, [el or 0 for el in eval_losses],
                 color="#e07b54", linestyle="--", linewidth=1.5, label="Val Loss")
        ax2.set_ylabel("Val Loss", color="#e07b54")
    axes[1].set_title("Validation Macro F1 per Epoch", fontsize=12, fontweight="bold")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Macro F1")
    axes[1].set_ylim(0, 1)
    axes[1].grid(alpha=0.3)
    axes[1].legend(loc="lower right")

plt.tight_layout()
plt.savefig(DATASET_OUT / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## Section 10 — Evaluate on Validation & Test Sets

Expected metrics (target range):
```
Macro F1:              0.85–0.89
Class 0 (surface) F1:  0.88–0.92
Class 1 (regular) F1:  0.90–0.93
Class 2 (deep) F1:     0.84–0.89
Class 3 (expert) F1:   0.75–0.82  ← fewest samples, hardest
```

In [ ]:
def evaluate_split(split_name: str, dataset: Dataset, df_split: pd.DataFrame):
    """Run inference on a dataset split and print full metrics + confusion matrix."""
    print(f"\n{'='*60}")
    print(f"  Evaluation: {split_name}")
    print(f"{'='*60}")

    predictions   = trainer.predict(dataset)
    pred_labels   = np.argmax(predictions.predictions, axis=-1)
    true_labels   = df_split["label"].values

    acc      = accuracy_score(true_labels, pred_labels)
    macro_f1 = f1_score(true_labels, pred_labels, average="macro", zero_division=0)

    print(f"\nOverall Accuracy : {acc*100:.2f}%")
    print(f"Macro F1         : {macro_f1:.4f}")

    print("\nPer-class Report:")
    print(classification_report(
        true_labels, pred_labels,
        target_names=CLASS_NAMES,
        zero_division=0
    ))

    # Confusion matrix
    cm = confusion_matrix(true_labels, pred_labels)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                linewidths=0.5, linecolor="grey")
    axes[0].set_title(f"{split_name} — Confusion Matrix (counts)", fontweight="bold")
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("True")
    axes[0].tick_params(axis="x", rotation=30)

    sns.heatmap(cm_pct, annot=True, fmt=".1f", cmap="Blues", ax=axes[1],
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                linewidths=0.5, linecolor="grey", vmin=0, vmax=100)
    axes[1].set_title(f"{split_name} — Confusion Matrix (%)", fontweight="bold")
    axes[1].set_xlabel("Predicted")
    axes[1].set_ylabel("True")
    axes[1].tick_params(axis="x", rotation=30)

    plt.tight_layout()
    plt.savefig(DATASET_OUT / f"confusion_matrix_{split_name.lower()}.png", dpi=150, bbox_inches="tight")
    plt.show()

    return {"accuracy": acc, "macro_f1": macro_f1, "predictions": pred_labels}


# ── Validation set ─────────────────────────────────────────────────────────────
val_results  = evaluate_split("Validation", val_ds, val_df)

# ── Test set ───────────────────────────────────────────────────────────────────
test_results = evaluate_split("Test", test_ds, test_df)

print(f"\n{'='*60}")
print(f"  Final Test Results")
print(f"  Accuracy : {test_results['accuracy']*100:.2f}%")
print(f"  Macro F1 : {test_results['macro_f1']:.4f}")
print(f"{'='*60}")

## Section 11 — Save Model

On **Kaggle**, the model is saved to `/kaggle/working/depth-modernbert/`. After training finishes, go to **Output** tab → download the folder as a zip, or add it as a Kaggle dataset for reuse.

> No Drive mounting needed — Kaggle persists `/kaggle/working/` as the notebook output automatically.

In [ ]:
# On Kaggle, MODEL_SAVE_PATH is already /kaggle/working/depth-modernbert/
# No Drive mounting required — Kaggle auto-persists /kaggle/working/ as output.
save_path = MODEL_SAVE_PATH
save_path.mkdir(parents=True, exist_ok=True)
print(f"Saving model to: {save_path}")

# Save model + tokenizer
trainer.save_model(str(save_path))
tokenizer.save_pretrained(str(save_path))

# Save depth label mapping for inference
with open(save_path / "depth_label_map.json", "w") as f:
    json.dump({str(k): v for k, v in INT_TO_DEPTH_FLOAT.items()}, f, indent=2)

# Save final metrics alongside the model
metrics_out = {
    "test_macro_f1":  round(test_results["macro_f1"], 4),
    "test_accuracy":  round(test_results["accuracy"], 4),
    "class_names":    CLASS_NAMES,
    "base_model":     BASE_MODEL_NAME,
}
with open(save_path / "metrics.json", "w") as f:
    json.dump(metrics_out, f, indent=2)

# Verify
saved_files = list(save_path.iterdir())
print(f"\n✓ Model saved. Files in {save_path}:")
for fpath in sorted(saved_files):
    size_kb = fpath.stat().st_size / 1024
    print(f"  {fpath.name:<35s} {size_kb:>8.1f} KB")

if IS_KAGGLE:
    print("\n→ On Kaggle: go to the Output tab to download or save as a dataset.")

## Section 12 — Inference & Runtime Integration

The `get_depth_score()` function is the drop-in replacement for the GPT mastery scoring call in the ConFit pipeline. Load once at startup, call per skill evidence sentence.

```
get_depth_score("optimized PySpark jobs processing 10TB daily")  → 0.8
get_depth_score("familiar with Python")                          → 0.2
get_depth_score("architected microservices platform at scale")   → 1.0
```

In [ ]:
# ── Load saved model for inference ────────────────────────────────────────────
_inf_tokenizer = AutoTokenizer.from_pretrained(str(save_path))
_inf_model     = AutoModelForSequenceClassification.from_pretrained(str(save_path))
_inf_model     = _inf_model.to(device)
_inf_model.eval()

_depth_map = {0: 0.2, 1: 0.5, 2: 0.8, 3: 1.0}


def get_depth_score(context_sentence: str) -> float:
    """
    Classify a skill evidence sentence into one of 4 depth classes
    and return the corresponding depth float score.

    Args:
        context_sentence: A sentence describing how a skill was used.

    Returns:
        float: 0.2 (surface), 0.5 (regular), 0.8 (deep), 1.0 (expert)
    """
    inputs = _inf_tokenizer(
        context_sentence,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding="max_length",
    ).to(device)

    with torch.no_grad():
        logits = _inf_model(**inputs).logits

    pred_class  = logits.argmax(dim=-1).item()
    confidence  = torch.softmax(logits, dim=-1).squeeze()[pred_class].item()
    depth_score = _depth_map[pred_class]

    return depth_score, confidence, pred_class


# ── Sanity-check on held-out examples ─────────────────────────────────────────
test_sentences = [
    # expected → 0.2 (surface)
    "familiar with Python and basic scripting",
    "coursework in Java and object-oriented programming",
    # expected → 0.5 (regular)
    "used SQL to query customer transactions and generate monthly reports",
    "implemented REST APIs using Django to serve mobile application data",
    # expected → 0.8 (deep)
    "optimized PySpark jobs processing 10TB of daily event logs, reducing runtime by 70%",
    "led migration of monolithic Rails app to microservices, improving deploy frequency 3x",
    # expected → 1.0 (expert)
    "architected the entire ML platform from scratch, serving 500M predictions/day",
    "core contributor to PyTorch distributed training module, authored 3 conference papers",
]

print(f"{'Sentence':<65s}  {'Score':>6s}  {'Class':<16s}  {'Conf':>6s}")
print("-" * 105)
for sent in test_sentences:
    score, conf, cls = get_depth_score(sent)
    class_label = CLASS_NAMES[cls]
    display = sent[:62] + "..." if len(sent) > 65 else sent
    print(f"{display:<65s}  {score:>6.1f}  {class_label:<16s}  {conf:>5.1%}")

In [ ]:
# ── Batch inference helper (for production use in the ConFit pipeline) ─────────

def get_depth_scores_batch(sentences: list[str], batch_size: int = 64) -> list[float]:
    """
    Batch inference — more efficient than calling get_depth_score() in a loop
    when processing many sentences at startup / during ingestion.

    Returns a list of depth floats (0.2 / 0.5 / 0.8 / 1.0) in the same order.
    """
    all_scores = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i + batch_size]
        inputs = _inf_tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_SEQ_LEN,
            padding=True,
        ).to(device)
        with torch.no_grad():
            logits = _inf_model(**inputs).logits
        pred_classes = logits.argmax(dim=-1).tolist()
        all_scores.extend([_depth_map[c] for c in pred_classes])
    return all_scores


# Demo: batch scoring on test sentences
batch_scores = get_depth_scores_batch(test_sentences)
print("Batch depth scores:")
for sent, score in zip(test_sentences, batch_scores):
    display = sent[:70] + "..." if len(sent) > 73 else sent
    print(f"  {score:.1f}  {display}")

print("\n✓ Notebook complete. Model ready for integration.")